# Fun

Purpose of this notebook is to actually do training to discover different model 
configurations that would work to train a pretty good TinyGPT. 

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


cuda
Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%rm -rf 242b-hw3
!git clone https://github.com/sanchitram1/242b-hw3
%cd 242b-hw3

Cloning into '242b-hw3'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 54 (delta 25), reused 45 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 469.41 KiB | 13.04 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/242b-hw3


In [4]:
from tokenizer import build_tokenizer

In [6]:
from config import (
    DataConfig,
    GlobalTrainingConfig,
    RunConfig,
    TokenConfig,
    TokenizationConfig,
)


token_config = TokenConfig()
data_config = DataConfig()
global_training_config = GlobalTrainingConfig()
tokenization_config = TokenizationConfig()
run_config = RunConfig("long-100000")

Directory already exists: /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/baseline-1000


In [8]:
tokenizer = build_tokenizer(
    tokenization_config, token_config, data_config.training_file_colab
)


In [10]:
from tokenizer import count_tokens, build_token_memmap

In [18]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)


In [ ]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)
valid_token_count = count_tokens(tokenization_config, tokenizer, data_config.validation_file_colab)
print(f"There are {train_token_count} tokens in train, {valid_token_count} valid")

train_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.training_file_colab,
    train_token_count,
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config, 
    token_config, 
    tokenizer, 
    data_config.validation_file_colab,
    valid_token_count
)

In [15]:
valid_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.validation_file_colab
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.validation_file_colab,
    valid_token_count,
)

In [16]:
from models import TokenChunkDataset

In [22]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, global_training_config.context_length
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, global_training_config.context_length
)

In [23]:
from config import ModelConfig

In [24]:
configs = [
    ModelConfig(
        name="small",
        d_model=128,
        n_heads=4,
        n_layers=3,
        d_ff=384,
        batch_size=16,
        learning_rate=3e-4,
        weight_decay=0.1,
        warmup_steps=50,
        max_steps=1000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="base",
        d_model=160,
        n_heads=5,
        n_layers=4,
        d_ff=480,
        batch_size=16,
        learning_rate=3e-4,
        weight_decay=0.1,
        warmup_steps=80,
        max_steps=1000,
        use_amp=torch.cuda.is_available(),
    ),
    ModelConfig(
        name="large",
        d_model=192,
        n_heads=6,
        n_layers=5,
        d_ff=576,
        batch_size=12,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=80,
        max_steps=1000,
        use_amp=torch.cuda.is_available(),
    ),
]


In [26]:
from training import train_model
import json
from utils import save_json
from plot import plot_training_curves, plot_validation_curves

In [30]:
global_training_config.context_window = global_training_config.context_length

In [31]:
results: dict[str, dict] = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for config in configs:
    metrics_path = run_config.metrics / f"{config.name}.json"
    if metrics_path.exists():
        results[config.name] = json.loads(metrics_path.read_text(encoding="utf-8"))
        continue
    result = train_model(
        run_config,
        global_training_config,
        config,
        tokenizer,
        train_dataset,
        valid_dataset,
        device=device,
    )
    results[config.name] = result
    save_json(result, metrics_path)

    if device.type == "cuda":
        torch.cuda.empty_cache()

plot_training_curves(results, run_config.plots / "training_loss_all_models.png")
plot_validation_curves(results, run_config.plots / "validation_loss_all_models.png")


[small] Loss(step=1)=8.03
[small] Loss(step=100)=6.58
[small] Loss(step=200)=6.36
[small] Loss(step=300)=6.46
[small] Loss(step=400)=6.45
[small] Loss(step=500)=6.46
[small] Loss(step=600)=6.56
[small] Loss(step=700)=6.42
[small] Loss(step=800)=6.43
[small] Loss(step=900)=6.41
[small] Loss(step=1000)=6.54
[base] Loss(step=1)=8.04
[base] Loss(step=100)=6.40
[base] Loss(step=200)=6.40
[base] Loss(step=300)=6.40
[base] Loss(step=400)=6.34
[base] Loss(step=500)=6.38
[base] Loss(step=600)=6.50
[base] Loss(step=700)=6.46
[base] Loss(step=800)=6.51
[base] Loss(step=900)=6.39
[base] Loss(step=1000)=6.52
[large] Loss(step=1)=8.03
[large] Loss(step=100)=6.49
[large] Loss(step=200)=6.37
[large] Loss(step=300)=6.45
[large] Loss(step=400)=6.51
[large] Loss(step=500)=6.44
[large] Loss(step=600)=6.40
[large] Loss(step=700)=6.46
[large] Loss(step=800)=6.60
[large] Loss(step=900)=6.46
[large] Loss(step=1000)=6.40
